In [ ]:
# ── Cell 1: pip installs + imports ──────────────────────────────────────────
# torch/torchvision are pre-installed on Kaggle — do NOT reinstall them.
# peft requires accelerate; install both together.
%pip install -q --no-deps datasets transformers huggingface_hub peft accelerate
%pip install -q --no-deps ipywidgets matplotlib scikit-learn

import os, math, glob as _glob, shutil
from itertools import islice

import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset as hf_load
import datasets as _ds
from huggingface_hub import login

# ── HuggingFace auth: Kaggle Secrets ────────────────────────────────────────
from kaggle_secrets import UserSecretsClient
login(token=UserSecretsClient().get_secret('HF_TOKEN'))
print('HuggingFace login: OK')

print('torch :', torch.__version__)
print('cuda  :', torch.cuda.is_available())
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

In [ ]:
# ── Cell 2: hyperparameters ───────────────────────────────────────────────────
MODEL_NAME = 'Qwen/Qwen2.5-0.5B'
NUM_LABELS = 5
MAX_LENGTH = 256

# LoRA
LORA_R           = 8
LORA_ALPHA       = 16
LORA_DROPOUT     = 0.1
LORA_TARGET_MODS = ['q_proj', 'k_proj', 'v_proj', 'o_proj']

# Training
BATCH_SIZE         = 32
ACCUMULATION_STEPS = 2
MAX_LR             = 2e-4   # higher LR is normal for LoRA vs full fine-tune
MIN_LR             = 1e-5
WEIGHT_DECAY       = 0.01
NUM_EPOCHS         = 3
WARMUP_RATIO       = 0.06

In [ ]:
# ── Cell 3: tokenizer + model + LoRA ─────────────────────────────────────────
print(f'Loading tokenizer: {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print(f'Loading model: {MODEL_NAME} ...')
# Qwen2.5 config.json sets torch_dtype=bfloat16; load explicitly in bfloat16.
# BFloat16 has the same exponent range as FP32 so GradScaler is unnecessary —
# the Trainer uses plain autocast(dtype=torch.bfloat16) with no scaler.
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    dtype=torch.bfloat16,
)
base_model.config.pad_token_id = tokenizer.pad_token_id

# modules_to_save=['score'] ensures the classification head is fully
# fine-tuned and saved alongside the LoRA adapters.
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODS,
    modules_to_save=['score'],
    bias='none',
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
model = model.to(DEVICE)

# ── sanity forward pass ───────────────────────────────────────────────────────
enc = tokenizer('This movie was fantastic!', return_tensors='pt',
                max_length=MAX_LENGTH, padding='max_length', truncation=True)
model.eval()
with torch.no_grad():
    out = model(enc['input_ids'].to(DEVICE), attention_mask=enc['attention_mask'].to(DEVICE))
assert out.logits.shape == (1, NUM_LABELS), f'Expected (1,{NUM_LABELS}), got {out.logits.shape}'
print('logits shape:', out.logits.shape)
print('forward pass OK')

In [ ]:
# ── Cell 4: AmazonReview Dataset + DataLoaders ───────────────────────────────
# Reuses the same HF Hub dataset cache as bert-finetune-kaggle.ipynb so no
# re-download is needed when both notebooks run in the same Kaggle account.

DATASET_REPO_ID          = 'cogsci13/Amazon-Reviews-2023-Books-Review'
DATASET_SPLIT            = 'full'
DATASET_LABEL            = 'books_review'
N_SAMPLES                = 1_500_000
HF_DATASET_CACHE_REPO_ID = 'AgentPhoenix7/SLM-project-dataset-cache'
CACHE_ROOT               = '/kaggle/working/data' if os.path.exists('/kaggle/working') else 'data'
DATASET_CACHE_NAME       = f'amazon_reviews_2023_{DATASET_LABEL}_{N_SAMPLES}'
DATASET_CACHE_DIR        = os.path.join(CACHE_ROOT, DATASET_CACHE_NAME)


class AmazonReview(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_length=256):
        self.dataset    = hf_dataset
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item   = self.dataset[idx]
        text   = item.get('text') or ''
        rating = item.get('rating')
        rating = rating if rating is not None else 3
        label  = max(0, min(4, int(float(rating)) - 1))

        tokens = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids':      tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
            'label':          torch.tensor(label, dtype=torch.long),
        }


def load_dataset_cache():
    if os.path.exists(DATASET_CACHE_DIR):
        print(f'Loading local dataset cache from {DATASET_CACHE_DIR}')
        return _ds.load_from_disk(DATASET_CACHE_DIR)

    if HF_DATASET_CACHE_REPO_ID:
        try:
            from huggingface_hub import snapshot_download
            print(f'Downloading dataset cache from HF Hub: {HF_DATASET_CACHE_REPO_ID}')
            os.makedirs(CACHE_ROOT, exist_ok=True)
            snapshot_download(
                repo_id=HF_DATASET_CACHE_REPO_ID,
                repo_type='dataset',
                allow_patterns=f'{DATASET_CACHE_NAME}/**',
                local_dir=CACHE_ROOT,
            )
            if os.path.exists(DATASET_CACHE_DIR):
                return _ds.load_from_disk(DATASET_CACHE_DIR)
            print('HF Hub dataset cache path not found; will stream source dataset.')
        except Exception as e:
            print(f'HF Hub dataset cache unavailable: {e}')
    return None


def save_dataset_cache(dataset):
    os.makedirs(CACHE_ROOT, exist_ok=True)
    tmp = DATASET_CACHE_DIR + '.tmp'
    if os.path.exists(tmp):
        shutil.rmtree(tmp)
    dataset.save_to_disk(tmp)
    os.rename(tmp, DATASET_CACHE_DIR)
    print(f'Saved local dataset cache to {DATASET_CACHE_DIR}')

    if HF_DATASET_CACHE_REPO_ID:
        try:
            from huggingface_hub import HfApi
            api = HfApi()
            api.create_repo(HF_DATASET_CACHE_REPO_ID, repo_type='dataset', exist_ok=True, private=True)
            api.upload_folder(
                folder_path=DATASET_CACHE_DIR,
                path_in_repo=DATASET_CACHE_NAME,
                repo_id=HF_DATASET_CACHE_REPO_ID,
                repo_type='dataset',
            )
            print(f'Uploaded dataset cache to HF Hub: {HF_DATASET_CACHE_REPO_ID}/{DATASET_CACHE_NAME}')
        except Exception as e:
            print(f'HF Hub dataset cache upload failed (non-fatal): {e}')


raw = load_dataset_cache()
if raw is None:
    print(f'Streaming {N_SAMPLES:,} samples from {DATASET_REPO_ID}/{DATASET_SPLIT} ...')
    stream = hf_load(DATASET_REPO_ID, split=DATASET_SPLIT, streaming=True)
    raw = _ds.Dataset.from_list(list(islice(stream, N_SAMPLES)))
    save_dataset_cache(raw)
print(f'Total samples : {len(raw):,}')
print('Columns       :', raw.column_names)

splits   = raw.train_test_split(test_size=0.1, seed=42)
train_ds = AmazonReview(splits['train'], tokenizer, max_length=MAX_LENGTH)
val_ds   = AmazonReview(splits['test'],  tokenizer, max_length=MAX_LENGTH)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
print(f'Train batches : {len(train_loader):,}')
print(f'Val   batches : {len(val_loader):,}')

batch = next(iter(train_loader))
print('Sample batch  :', {k: tuple(v.shape) for k, v in batch.items()})

In [ ]:
# ── Cell 5: training infrastructure ──────────────────────────────────────────
# CKPT_EVERY: how many optimizer steps between mid-epoch checkpoint + hub push.
# 1.5 M samples, batch 32, accum 2 → ~21 000 optimizer steps/epoch → 500 is fine.
CKPT_EVERY      = 500
MAX_VAL_BATCHES = 200   # cap mid-epoch validation to ~6 400 samples


class CosineScheduler:
    def __init__(self, optimizer, warmup_steps, total_steps, max_lr, min_lr):
        self.optimizer    = optimizer
        self.warmup_steps = warmup_steps
        self.total_steps  = total_steps
        self.max_lr       = max_lr
        self.min_lr       = min_lr
        self.current_step = 0

    def get_lr(self):
        if self.current_step >= self.total_steps:
            return self.min_lr
        if self.current_step < self.warmup_steps:
            return self.max_lr * self.current_step / self.warmup_steps
        progress = (self.current_step - self.warmup_steps) / max(1, self.total_steps - self.warmup_steps)
        return self.min_lr + 0.5 * (self.max_lr - self.min_lr) * (1 + math.cos(math.pi * progress))

    def step(self):
        for pg in self.optimizer.param_groups:
            pg['lr'] = self.get_lr()
        self.current_step += 1

    def state_dict(self):
        return {k: v for k, v in self.__dict__.items() if k != 'optimizer'}

    def load_state_dict(self, sd):
        for k, v in sd.items():
            setattr(self, k, v)


def configure_optimizer(model, lr, weight_decay):
    decay, no_decay = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if 'bias' in name or 'norm' in name or 'LayerNorm' in name:
            no_decay.append(param)
        else:
            decay.append(param)
    return AdamW(
        [{'params': decay,    'weight_decay': weight_decay},
         {'params': no_decay, 'weight_decay': 0.0}],
        lr=lr, betas=(0.9, 0.999), eps=1e-8,
    )


class Trainer:
    def __init__(self, model, train_loader, val_loader, scheduler,
                 optimizer=None, device='cpu', use_amp=False,
                 accumulation_steps=1, checkpoint_dir='./checkpoints',
                 hf_repo_id=None):
        self.model              = model
        self.train_loader       = train_loader
        self.val_loader         = val_loader
        self.optimizer          = optimizer
        self.scheduler          = scheduler
        self.device             = device
        # Qwen2.5 is bfloat16; GradScaler is only for float16.
        # use_amp still enables autocast(dtype=torch.bfloat16) for speed.
        self.use_amp            = use_amp and device != 'cpu'
        self.accumulation_steps = accumulation_steps
        self.checkpoint_dir     = checkpoint_dir
        self.hf_repo_id         = hf_repo_id
        self.global_step        = 0
        self.best_val_loss      = float('inf')
        os.makedirs(checkpoint_dir, exist_ok=True)

        if not hf_repo_id:
            print('WARNING: hf_repo_id not set — checkpoints will not survive session end.')

        if torch.cuda.device_count() > 1:
            self.model = torch.nn.DataParallel(self.model)

    def _unwrap(self):
        return self.model.module if isinstance(self.model, torch.nn.DataParallel) else self.model

    # ── single epoch ─────────────────────────────────────────────────────────
    def train_epoch(self, epoch):
        self.model.train()
        total_loss = 0.0
        step = -1

        for step, batch in enumerate(self.train_loader):
            input_ids      = batch['input_ids'].to(self.device)
            attention_mask = batch['attention_mask'].to(self.device)
            labels         = batch['label'].to(self.device)

            if self.use_amp:
                with autocast(device_type='cuda', dtype=torch.bfloat16):
                    out  = self.model(input_ids, attention_mask=attention_mask)
                    loss = F.cross_entropy(out.logits, labels) / self.accumulation_steps
            else:
                out  = self.model(input_ids, attention_mask=attention_mask)
                loss = F.cross_entropy(out.logits, labels) / self.accumulation_steps
            loss.backward()

            if (step + 1) % self.accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.optimizer.step()
                self.optimizer.zero_grad()
                self.scheduler.step()
                self.global_step += 1

                if self.global_step % CKPT_EVERY == 0:
                    metrics = self.validate(max_batches=MAX_VAL_BATCHES)
                    self.model.train()
                    tag = f'step_{self.global_step}'
                    print(f'[step {self.global_step}]  '
                          f'val_loss={metrics["val_loss"]:.4f}  '
                          f'val_acc={metrics["accuracy"]:.4f}')
                    self.save_checkpoint(
                        epoch, metrics['val_loss'],
                        os.path.join(self.checkpoint_dir, f'{tag}.pt'),
                        checkpoint_type='step',
                    )
                    self._prune_step_checkpoints(keep=3)
                    if metrics['val_loss'] < self.best_val_loss:
                        self.best_val_loss = metrics['val_loss']
                        self.save_checkpoint(
                            epoch, metrics['val_loss'],
                            os.path.join(self.checkpoint_dir, 'best.pt'),
                            checkpoint_type='step',
                        )
                        print(f'  ** new best at step {self.global_step}: '
                              f'val_loss={self.best_val_loss:.4f} **')

            total_loss += loss.item() * self.accumulation_steps
            if step % 100 == 0:
                lr = self.optimizer.param_groups[0]['lr']
                print(f'  epoch {epoch}  step {step:>6d}  '
                      f'loss {loss.item() * self.accumulation_steps:.4f}  lr {lr:.2e}')

        if (step + 1) % self.accumulation_steps != 0:
            self.optimizer.zero_grad()
        return total_loss / max(1, len(self.train_loader))

    # ── validation ───────────────────────────────────────────────────────────
    def validate(self, max_batches=None):
        self.model.eval()
        total_loss, correct, total, n_batches = 0.0, 0, 0, 0
        limit = max_batches or len(self.val_loader)
        print(f'  [validate] running up to {limit} batches ...', flush=True)
        with torch.no_grad():
            for i, batch in enumerate(self.val_loader):
                if i >= limit:
                    break
                input_ids      = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels         = batch['label'].to(self.device)
                out        = self.model(input_ids, attention_mask=attention_mask)
                total_loss += F.cross_entropy(out.logits, labels).item()
                correct    += (out.logits.argmax(-1) == labels).sum().item()
                total      += labels.size(0)
                n_batches  += 1
        print('  [validate] done.', flush=True)
        return {'val_loss': total_loss / max(1, n_batches),
                'accuracy': correct / max(1, total)}

    # ── full training loop ────────────────────────────────────────────────────
    def train(self, num_epochs, start_epoch=0):
        for epoch in range(start_epoch, num_epochs):
            print(f'\n{"="*60}  Epoch {epoch+1}/{num_epochs}  {"="*60}')
            train_loss = self.train_epoch(epoch)
            metrics    = self.validate()
            self.model.train()
            print(f'  [epoch {epoch+1} end]  '
                  f'train_loss={train_loss:.4f}  '
                  f'val_loss={metrics["val_loss"]:.4f}  '
                  f'val_acc={metrics["accuracy"]:.4f}')
            self.save_checkpoint(
                epoch, metrics['val_loss'],
                os.path.join(self.checkpoint_dir, f'epoch_{epoch+1}.pt'),
                checkpoint_type='epoch',
            )
            if metrics['val_loss'] < self.best_val_loss:
                self.best_val_loss = metrics['val_loss']
                self.save_checkpoint(
                    epoch, metrics['val_loss'],
                    os.path.join(self.checkpoint_dir, 'best.pt'),
                    checkpoint_type='epoch',
                )
                print('  ** new best (epoch end) **')
        return self.best_val_loss

    # ── checkpointing ─────────────────────────────────────────────────────────
    # Only trainable params are saved (LoRA adapters + score head) — much
    # smaller than the full model state (~10–30 MB vs ~1 GB).
    def save_checkpoint(self, epoch, val_loss, path, checkpoint_type='step'):
        m = self._unwrap()
        trainable_sd = {
            k: v.detach().cpu()
            for k, v in m.named_parameters()
            if v.requires_grad
        }
        state = {
            'epoch':                epoch,
            'global_step':          self.global_step,
            'checkpoint_type':      checkpoint_type,
            'trainable_state_dict': trainable_sd,
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'val_loss':             val_loss,
            'best_val_loss':        self.best_val_loss,
        }
        torch.save(state, path)

        last_path = os.path.join(self.checkpoint_dir, 'last.pt')
        torch.save(state, last_path)

        if self.hf_repo_id:
            self._push_to_hub(last_path)
            if os.path.basename(path) == 'best.pt':
                self._push_to_hub(path, 'checkpoints/best.pt')

    def _push_to_hub(self, local_path, repo_path='checkpoints/last.pt'):
        try:
            from huggingface_hub import HfApi
            api = HfApi()
            api.create_repo(self.hf_repo_id, repo_type='model', exist_ok=True, private=True)
            api.upload_file(
                path_or_fileobj=local_path,
                path_in_repo=repo_path,
                repo_id=self.hf_repo_id,
                repo_type='model',
            )
            print(f'  [hub] ✓ {repo_path} → {self.hf_repo_id} '
                  f'(step {self.global_step})', flush=True)
        except Exception as e:
            print(f'  [hub] upload failed (non-fatal): {e}', flush=True)

    def _prune_step_checkpoints(self, keep=3):
        step_ckpts = sorted(_glob.glob(os.path.join(self.checkpoint_dir, 'step_*.pt')))
        for old in step_ckpts[:-keep]:
            os.remove(old)

    def load_checkpoint(self, path):
        ckpt = torch.load(path, map_location=self.device, weights_only=False)
        m = self._unwrap()
        current_sd = m.state_dict()
        for k, v in ckpt['trainable_state_dict'].items():
            if k in current_sd:
                current_sd[k].copy_(v.to(self.device))
        m.load_state_dict(current_sd)
        self.optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        self.scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        self.global_step   = ckpt.get('global_step', 0)
        self.best_val_loss = ckpt.get('best_val_loss', float('inf'))
        ckpt_type = ckpt.get('checkpoint_type', 'epoch')
        print(f'Resumed from {os.path.basename(path)}  '
              f'(epoch={ckpt["epoch"]}, step={self.global_step}, '
              f'type={ckpt_type}, best_val_loss={self.best_val_loss:.4f})')
        return ckpt['epoch'] + (1 if ckpt_type == 'epoch' else 0)


print('Training infrastructure defined.')
print(f'CKPT_EVERY = {CKPT_EVERY} optimizer steps')

In [ ]:
# ── Cell 6: run training ──────────────────────────────────────────────────────
CKPT_DIR = './checkpoints'

# ── REQUIRED for cross-session resume ─────────────────────────────────────────
# /kaggle/working is wiped when a session ends. The HF Hub repo below is the
# only durable store. Set HF_REPO_ID to your repo; it is created automatically
# as private if it does not exist. Re-running all cells resumes automatically.
HF_REPO_ID = 'AgentPhoenix7/SLM-project-qwen'

if not HF_REPO_ID:
    print('WARNING: HF_REPO_ID not set — checkpoints will be lost on session end.')

TOTAL_STEPS  = NUM_EPOCHS * (len(train_loader) // ACCUMULATION_STEPS)
WARMUP_STEPS = int(WARMUP_RATIO * TOTAL_STEPS)

optimizer = configure_optimizer(model, lr=MAX_LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineScheduler(optimizer, WARMUP_STEPS, TOTAL_STEPS, MAX_LR, MIN_LR)
trainer   = Trainer(
    model, train_loader, val_loader, scheduler,
    optimizer=optimizer, device=DEVICE, use_amp=True,
    accumulation_steps=ACCUMULATION_STEPS,
    checkpoint_dir=CKPT_DIR, hf_repo_id=HF_REPO_ID,
)

# ── resume logic (same pattern as bert-finetune-kaggle.ipynb) ─────────────────
import glob
start_epoch = 0
resumed     = False

# 1. Try HF Hub first — only store that survives session end
if HF_REPO_ID and not resumed:
    try:
        from huggingface_hub import hf_hub_download
        hub_path = hf_hub_download(
            repo_id=HF_REPO_ID,
            filename='checkpoints/last.pt',
            repo_type='model',
            force_download=True,
        )
        start_epoch = trainer.load_checkpoint(hub_path)
        print(f'Resumed from HF Hub ({HF_REPO_ID}), start_epoch={start_epoch}')
        resumed = True
    except Exception as e:
        print(f'HF Hub: no checkpoint found ({e})')

# 2. Fall back to local last.pt (same-session kernel restart)
if not resumed:
    local_last = os.path.join(CKPT_DIR, 'last.pt')
    if os.path.exists(local_last):
        start_epoch = trainer.load_checkpoint(local_last)
        print(f'Resumed from local last.pt, start_epoch={start_epoch}')
        resumed = True

# 3. Fall back to latest epoch checkpoint
if not resumed:
    epoch_ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, 'epoch_*.pt')))
    if epoch_ckpts:
        start_epoch = trainer.load_checkpoint(epoch_ckpts[-1])
        print(f'Resumed from {epoch_ckpts[-1]}, start_epoch={start_epoch}')
        resumed = True

if not resumed:
    print('No checkpoint found — starting fresh.')

best_loss = trainer.train(NUM_EPOCHS, start_epoch=start_epoch)
print('Best val loss:', best_loss)

In [ ]:
# ── Cell 7: results / confusion matrix ───────────────────────────────────────
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, mean_absolute_error

trainer.load_checkpoint(os.path.join(CKPT_DIR, 'best.pt'))
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        ids   = batch['input_ids'].to(DEVICE)
        mask  = batch['attention_mask'].to(DEVICE)
        preds = model(ids, attention_mask=mask).logits.argmax(-1).cpu().tolist()
        all_preds  += preds
        all_labels += batch['label'].tolist()

cm   = confusion_matrix(all_labels, all_preds, labels=[0, 1, 2, 3, 4])
disp = ConfusionMatrixDisplay(cm, display_labels=[1, 2, 3, 4, 5])
disp.plot(cmap='Blues')
plt.title('Qwen2.5-0.5B Amazon Review — Confusion Matrix')
plt.tight_layout()
plt.savefig(os.path.join(CKPT_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

acc = accuracy_score(all_labels, all_preds)
mae = mean_absolute_error(all_labels, all_preds)
print(f'Accuracy : {acc:.4f}')
print(f'MAE      : {mae:.4f}')